# LongFlow — quality night: size ladder on capture v3

Runtime: **A100 GPU**, ~4–5 h after capture v3 completes. Pre-registration:
NOTES "QUALITY NIGHT" (2026-08-19). Requires `longflow_p1_cache_v3` on
Drive (capture_v3_colab.ipynb finished, manifest present).

Two heads, same recipe, on the 1.92M-frame clean pool: **width-640
(16.6M, the incumbent)** and **width-960 (~36M, first rung of the
capacity ladder — constraint-5 amendment)**. 40K steps (pool is 4×; ~21
epochs ≈ E3's sweet-spot epoch count), ckpts every 5K, overfit watch on.

Eval: held-out teacher-forced (heun8+CFG) per checkpoint + **30 s-chunked
closed-loop renders** for teacher / cleanabl (controls) / both new heads —
scored with the EAR PACK. Bars: DATA PAYS = +1 dB golden-window HNR over
cleanabl; SIZE PAYS = 960 ≥ 640 + 1 dB; WIN = half the 4 dB teacher gap
closed AND Josh's ear ≥ current best.

| cell | what |
|---|---|
| 1 | cold start (bulk-copy v3 cache local — ~15 GB, be patient) |
| 2 | pools + filename-bin held-out split |
| 3 | GATE: 640 @5K + 4 clips → Drive → LISTEN (constraint 6) |
| 4 | train both arms, 40K steps, ckpts every 5K |
| 5 | held-out renders per ckpt per arm |
| 6 | 30s-chunked closed loop: teacher / cleanabl / 640 / 960 |
| 7 | bundle → Drive root `quality_eval.zip` |


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
NOTEBOOK_VERSION = "Quality ladder v1.0 (2026-08-19): 640 vs 960 on capture v3"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
from pathlib import Path
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone failed — check repo access"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.cache.capture import load_utterance
from src.flow_head.cfm import heun_sample
from src.flow_head.integration import CFGFlowHeadPatch, _CFGField
from src.flow_head.model import FlowHead, FlowHeadConfig
from src.flow_head.trainer import load_checkpoint, pairs_from_files, train

CACHE_V3_DRIVE = "/content/drive/MyDrive/longflow_p1_cache_v3"
CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
TRAIN_CACHE_V1 = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
OUT = "/content/quality"
DRIVE_OUT = "/content/drive/MyDrive/longflow_quality"
EVAL_DIR = "/content/quality_eval"
for d in (OUT, DRIVE_OUT, EVAL_DIR, f"{DRIVE_OUT}/gate"):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{CACHE_V3_DRIVE}/capture_v3_manifest.json"), \
    "capture v3 not finished — run capture_v3_colab.ipynb to completion first"

LOCAL_CACHE = "/content/cache_v3"
n_drive = len(glob.glob(f"{CACHE_V3_DRIVE}/*.pt"))
if not os.path.exists(LOCAL_CACHE) or len(glob.glob(f"{LOCAL_CACHE}/*.pt")) < n_drive:
    os.makedirs(LOCAL_CACHE, exist_ok=True)
    print(f"bulk-copying {n_drive} v3 files (~15 GB) to local disk — several minutes...", flush=True)
    !cp {CACHE_V3_DRIVE}/*.pt {LOCAL_CACHE}/
print(f"{len(glob.glob(f'{LOCAL_CACHE}/*.pt'))} cache files local")

LOCAL_CKPT = "/content/ckpts"
os.makedirs(LOCAL_CKPT, exist_ok=True)
if not os.path.exists(f"{LOCAL_CKPT}/cleanabl_20k_step20000.pt"):
    shutil.copy(f"{CKPT_DIR}/cleanabl_20k_step20000.pt", LOCAL_CKPT)
head_cl, mean_c, std_c = load_checkpoint(f"{LOCAL_CKPT}/cleanabl_20k_step20000.pt")
head_cl = head_cl.to("cuda")

if os.path.exists(f"{DRIVE_OUT}/quality_report.json"):
    with open(f"{DRIVE_OUT}/quality_report.json") as f:
        report = json.load(f)
    print(f"resuming: {len(report['runs'])} runs already recorded")
else:
    report = {"notebook_version": NOTEBOOK_VERSION, "runs": []}

def done(tag):
    return os.path.exists(f"{DRIVE_OUT}/{tag}.wav")

def save_wav(tag, wav, meta, sub=""):
    d = f"{DRIVE_OUT}/{sub}" if sub else DRIVE_OUT
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{d}/{tag}.wav")
    report["runs"] = [r for r in report["runs"] if r.get("tag") != tag] + [{"tag": tag, **meta}]
    with open(f"{DRIVE_OUT}/quality_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print(f"saved {tag}: {len(wav)/24000:.1f}s  {meta}", flush=True)

def gen_inputs(texts, prompt_lists):
    inputs = processor(text=texts, voice_samples=prompt_lists,
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def decode_latents(z, chunk_frames=225):
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi
    wavs, shape_fn = [], None
    for i in range(0, z.shape[0], chunk_frames):
        chunk = z[i : i + chunk_frames]
        candidates = [shape_fn] if shape_fn is not None else [
            lambda c: c.unsqueeze(0), lambda c: c.unsqueeze(0).transpose(1, 2)
        ]
        decoded = None
        for fn in candidates:
            try:
                out = model.model.acoustic_tokenizer.decode(fn(chunk))
                decoded = out[0] if isinstance(out, tuple) else out
                shape_fn = fn
                break
            except Exception as e:
                print(f"decode attempt failed: {repr(e)[:120]}")
        if decoded is None:
            raise RuntimeError("both decode shapes failed — paste the errors to Claude")
        wavs.append(decoded.detach().float().cpu().numpy().squeeze())
        del out, decoded
        torch.cuda.empty_cache()
    return np.concatenate(wavs) if len(wavs) > 1 else wavs[0]

def flow_cfg_render(head, mean_t, std_t, utt, seed=0):
    field = _CFGField(head, utt.neg_hidden.float().cuda(), 1.3)
    g = torch.Generator(device="cuda").manual_seed(seed)
    z = heun_sample(field, utt.hidden.float().cuda(), head.cfg.d_latent,
                    nfe=8, sway=0.0, generator=g)
    return decode_latents(z * std_t.cuda() + mean_t.cuda())

print("READY")


In [ ]:
# ===== Pool + FILENAME-bin held-out split (5/bin, cv3_ files) =====
HELD_OUT_PER_BIN = 5
all_files = sorted(glob.glob(f"{LOCAL_CACHE}/*.pt"))

def fname_bin(path):
    return int(Path(path).stem.split("_")[1].rstrip("w"))

by_bin = {}
for f in all_files:
    by_bin.setdefault(fname_bin(f), []).append(f)
held_out_by_bin = {b: fs[:HELD_OUT_PER_BIN] for b, fs in sorted(by_bin.items())}
held_out_files = [f for fs in held_out_by_bin.values() for f in fs]
train_files = [f for b, fs in sorted(by_bin.items()) for f in fs[HELD_OUT_PER_BIN:]]
print({b: len(fs) for b, fs in sorted(by_bin.items())})
print(f"held-out {len(held_out_files)} / train {len(train_files)} scripts")
with open(f"{DRIVE_OUT}/quality_held_out_manifest.json", "w") as f:
    json.dump({"held_out": [Path(p).name for p in held_out_files]}, f, indent=2)

t0 = time.time()
data = pairs_from_files(train_files, dual_stream=False)  # cond-only training (arm-B recipe);
                                                          # neg is loaded per-utt at eval only
print(f"pool: {data.hidden.shape[0]} frames  d_model={data.d_model}  "
      f"d_latent={data.d_latent}  ({time.time()-t0:.0f}s to load)")


## 3. GATE — hard constraint 6 (LISTEN before cell 4)

5K steps of the 640 head on the v3 pool, 4 clips to
`Drive/longflow_quality/gate/`. Expected: shaky-but-intelligible (the
known 5K texture). Silence/collapse = STOP, paste findings.


In [ ]:
if glob.glob(f"{DRIVE_OUT}/gate/*_gate.wav"):
    print("gate clips already on Drive — skip to listening / cell 4")
else:
    gate_head = FlowHead(FlowHeadConfig(d_model=data.d_model, d_latent=data.d_latent))
    out = train(gate_head, data, steps=5000, batch_size=1024, lr=2e-4,
                ema_decay=0.999, device="cuda", log_every=1000)
    out["ema"].copy_to(gate_head)
    for fpath in (held_out_by_bin[150][0], held_out_by_bin[1200][0]):
        utt = load_utterance(fpath)
        sf.write(f"{DRIVE_OUT}/gate/{utt.utt_id}_teacher.wav",
                 decode_latents(utt.latent.float()), 24000)
        wav = flow_cfg_render(gate_head, data.mean, data.std, utt, seed=0)
        sf.write(f"{DRIVE_OUT}/gate/{utt.utt_id}_gate.wav", wav, 24000)
        print(f"gate clips for {utt.utt_id} on Drive", flush=True)
    del gate_head
    torch.cuda.empty_cache()
    print("\nLISTEN NOW (Drive/longflow_quality/gate/) — run cell 4 only on a PASS.")


In [ ]:
# ===== Train the ladder: 640 and 960, 40K steps, ckpts every 5K =====
ARMS = [("v3_640", dict()), ("v3_960", dict(width=960))]
for tag, kw in ARMS:
    final = f"{CKPT_DIR}/{tag}_step40000.pt"
    if os.path.exists(final):
        print(f"{tag}: final checkpoint already on Drive — skipping")
        continue
    head = FlowHead(FlowHeadConfig(d_model=data.d_model, d_latent=data.d_latent, **kw))
    print(f"=== {tag}: {head.param_count()/1e6:.2f}M params ===")
    t0 = time.time()
    train(head, data, steps=40000, batch_size=1024, lr=2e-4, lr_final=2e-5,
          ema_decay=0.9999, device="cuda", log_every=2000,
          checkpoint_every=5000,
          checkpoint_path_fn=lambda s, tag=tag: f"{CKPT_DIR}/{tag}_step{s}.pt")
    print(f"{tag} done in {(time.time()-t0)/60:.1f} min")
    del head
    torch.cuda.empty_cache()


In [ ]:
# ===== Held-out teacher-forced renders per checkpoint per arm =====
SUBSET_PER_BIN = 2
FULL_EVAL_STEPS = {40000}
manifest = {"held_out_per_bin": HELD_OUT_PER_BIN, "teacher": {}, "checkpoints": {}}

for fpath in held_out_files:
    utt = load_utterance(fpath)
    tname = f"{utt.utt_id}_teacher.wav"
    if not os.path.exists(f"{EVAL_DIR}/{tname}"):
        sf.write(f"{EVAL_DIR}/{tname}", decode_latents(utt.latent.float()), 24000)
    manifest["teacher"][utt.utt_id] = {"audio": tname, "text": utt.text,
                                       "target_words": fname_bin(fpath)}
print(f"{len(manifest['teacher'])} teacher references rendered")

subset_files = [fs[:SUBSET_PER_BIN] for fs in held_out_by_bin.values()]
subset_files = [f for fs in subset_files for f in fs]

for step in (5000, 10000, 15000, 20000, 25000, 30000, 35000, 40000):
    for tag in ("v3_640", "v3_960"):
        key = f"{step}:{tag}"
        if key in manifest["checkpoints"]:
            continue
        p = f"{CKPT_DIR}/{tag}_step{step}.pt"
        if not os.path.exists(p):
            continue
        head, mean_t, std_t = load_checkpoint(p)
        head = head.to("cuda")
        eval_files = held_out_files if step in FULL_EVAL_STEPS else subset_files
        entries = []
        for fpath in eval_files:
            utt = load_utterance(fpath)
            name = f"{utt.utt_id}_{tag}_step{step}.wav"
            if not os.path.exists(f"{EVAL_DIR}/{name}"):
                sf.write(f"{EVAL_DIR}/{name}",
                         flow_cfg_render(head, mean_t, std_t, utt, seed=0), 24000)
            entries.append({"utt_id": utt.utt_id, "audio": name,
                            "teacher_audio": manifest["teacher"][utt.utt_id]["audio"],
                            "text": utt.text, "target_words": fname_bin(fpath), "arm": tag})
        manifest["checkpoints"][key] = entries
        del head
        torch.cuda.empty_cache()
        print(f"{key}: {len(entries)} renders", flush=True)

with open(f"{EVAL_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("held-out eval rendered")


## 6. 30 s-chunked closed loop — the product-shape test

Same 803-word script as every closed loop, now in **~80-word chunks
(~30 s — inside the golden window per the forensics prescription)**,
0.25 s crossfades. Four engines: teacher, cleanabl (the incumbent
control), v3_640, v3_960 — all seed-matched per chunk.


In [ ]:
def drive_glob(pattern, tries=4, wait=15):
    for i in range(tries):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits
        print(f"empty listing for {pattern} — retry {i+1}/{tries} in {wait}s", flush=True)
        time.sleep(wait)
    raise RuntimeError(f"still empty after {tries} tries: {pattern}")

V1_LOCAL = "/content/v1texts"
if len(glob.glob(f"{V1_LOCAL}/*.pt")) < 300:
    os.makedirs(V1_LOCAL, exist_ok=True)
    for f in drive_glob(f"{TRAIN_CACHE_V1}/*.pt")[-300:]:
        shutil.copy(f, V1_LOCAL)
sents = []
for f in sorted(glob.glob(f"{V1_LOCAL}/*.pt")):
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
P0 = drive_glob(f"{EVAL_CACHE_DIR}/*_prompt.wav")[0]

def turnscript_turns(sentences, target=60, speaker=1):
    turns, cur, w = [], [], 0
    for s in sentences:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append(f"Speaker {speaker}: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return turns

ABL_WORDS, w = [], 0
for s in pool:
    ABL_WORDS.append(s); w += len(s.split())
    if w >= 800:
        break
TURNS = turnscript_turns(ABL_WORDS)
CHUNKS, cur, cw = [], [], 0
for t in TURNS:
    cur.append(t); cw += len(t.split())
    if cw >= 80:  # ~30s @ natural rate — the forensics prescription
        CHUNKS.append("\n".join(cur) + "\n"); cur, cw = [], 0
if cur:
    CHUNKS.append("\n".join(cur) + "\n")
print(f"{sum(len(t.split()) for t in TURNS)} words -> {len(CHUNKS)} chunks")
report["cl_script"] = "\n".join(TURNS) + "\n"
report["chunks_n"] = len(CHUNKS)

def crossfade_stitch(wavs, sr=24000, fade_s=0.25):
    n = int(sr * fade_s)
    out = wavs[0]
    for wv in wavs[1:]:
        if len(out) < n or len(wv) < n:
            out = np.concatenate([out, wv])
            continue
        fade = np.linspace(0, 1, n, dtype=np.float32)
        out[-n:] = out[-n:] * (1 - fade) + wv[:n] * fade
        out = np.concatenate([out, wv[n:]])
    return out

heads = {}
for tag in ("v3_640", "v3_960"):
    h, mn, sd = load_checkpoint(f"{CKPT_DIR}/{tag}_step40000.pt")
    heads[tag] = (h.to("cuda"), mn, sd)
heads["cleanabl"] = (head_cl, mean_c, std_c)

ENGINES = [("qc_teacher", None), ("qc_cleanabl", "cleanabl"),
           ("qc_640", "v3_640"), ("qc_960", "v3_960")]
for tag, hk in ENGINES:
    if done(tag):
        print(f"{tag}: already on Drive — skipping")
        continue
    chunk_wavs = []
    for ci, chunk_text in enumerate(CHUNKS):
        torch.manual_seed(ci)
        if hk is None:
            with torch.inference_mode():
                gen = model.generate(**gen_inputs([chunk_text], [[P0]]),
                                     tokenizer=processor.tokenizer,
                                     cfg_scale=1.3, max_new_tokens=600)
        else:
            h, mn, sd = heads[hk]
            with CFGFlowHeadPatch(model, h, mn, sd, nfe=8, sway=0.0,
                                  sampler=heun_sample) as patch, torch.inference_mode():
                gen = model.generate(**gen_inputs([chunk_text], [[P0]]),
                                     tokenizer=processor.tokenizer,
                                     cfg_scale=1.3, max_new_tokens=600)
        chunk_wavs.append(gen.speech_outputs[0].detach().float().cpu().numpy().squeeze())
        print(f"  {tag} chunk {ci+1}/{len(CHUNKS)}", flush=True)
    save_wav(tag, crossfade_stitch(chunk_wavs), {"engine": hk or "teacher",
                                                 "chunks": len(CHUNKS)})
print("chunked renders done")


In [ ]:
# ===== Bundle -> Drive root =====
import zipfile
with open(f"{EVAL_DIR}/quality_report.json", "w") as f:
    json.dump(report, f, indent=2)
ZIP = "/content/drive/MyDrive/quality_eval.zip"
with zipfile.ZipFile(ZIP, "w") as z:
    for f in os.listdir(EVAL_DIR):
        z.write(f"{EVAL_DIR}/{f}", f)
    for f in os.listdir(DRIVE_OUT):
        if f.endswith(".wav"):
            z.write(f"{DRIVE_OUT}/{f}", f"closed_loop/{f}")
print(f"bundle at {ZIP} ({os.path.getsize(ZIP)/1e9:.2f} GB) — run score_quality_gpu_colab.ipynb next")
